In [1]:
import scarf
scarf.configure_output(level="WARNING", progress=False)

dataset = scarf.cytebase.connect("scarf_docs").download_dataset(
    "tenx_5K_pbmc_rnaseq",
    destination="scarf_datasets",
    zarr=True,
)
ds = scarf.DataStore(f"{dataset}/data.zarr", nthreads=4)
baseline_run = ds.pipeline.open(label="docs_default")
cell_selection = baseline_run["analysis_cell_selection"]
hvg_ref = baseline_run["highly_variable_features"]

Downloading bucket files: 56073178 / 56073178 complete

Downloading bytes: 56073178 / 56073178 complete

In [2]:
normalized = baseline_run["normalized"]
pca = baseline_run["pca"]
ann = baseline_run["ann_index"]
neighbors_k11 = baseline_run["neighbors"]
graph_k11 = baseline_run["connectivity_map"]

In [3]:
[reopened_graph] = ds.list_artifacts(
    from_assay="RNA",
    kind="connectivity_map",
    operation="build_connectivity_map",
    inputs={"neighbors": neighbors_k11},
    complete_only=True,
)
assert reopened_graph == graph_k11

status = ds.inspect_artifact(reopened_graph)
{
    "operation": status.operation,
    "parameters": status.parameters,
    "inputs": status.inputs,
    "complete": status.complete,
}

{'operation': 'build_connectivity_map',
 'parameters': {'local_connectivity': 1.0, 'bandwidth': 1.5},
 'inputs': {'neighbors': {'type': 'artifact',
   'scope': 'assay',
   'kind': 'neighbors',
   'artifact_id': '63cd968f45f8b476dd3dd99b86111bb246ab8f15ca9d8fb16c2d95f518fdafa3',
   'assay': 'RNA'}},
 'complete': True}

In [4]:
neighbors_k15 = ds.query_neighbors(ann, k=15)
graph_k15 = ds.build_connectivity_map(neighbors_k15)

{
    "normalization reused": ds.run_normalization(cell_selection, hvg_ref) == normalized,
    "PCA reused": ds.run_pca(normalized, dims=15) == pca,
    "ANN index reused": ds.build_ann_index(pca) == ann,
    "neighbors recomputed": neighbors_k15 != neighbors_k11,
    "graph recomputed": graph_k15 != graph_k11,
}

{'normalization reused': True,
 'PCA reused': True,
 'ANN index reused': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [5]:
pca_dims20 = ds.run_pca(normalized, dims=20)
ann_dims20 = ds.build_ann_index(pca_dims20)
neighbors_dims20 = ds.query_neighbors(ann_dims20, k=11)
graph_dims20 = ds.build_connectivity_map(neighbors_dims20)

{
    "PCA recomputed": pca_dims20 != pca,
    "ANN index recomputed": ann_dims20 != ann,
    "neighbors recomputed": neighbors_dims20 != neighbors_k11,
    "graph recomputed": graph_dims20 != graph_k11,
}

{'PCA recomputed': True,
 'ANN index recomputed': True,
 'neighbors recomputed': True,
 'graph recomputed': True}

In [6]:
lineage = ds.lineage(
    {
        "k11 graph": graph_k11,
        "k15 graph": graph_k15,
        "dims20 graph": graph_dims20,
    }
)
lineage

```mermaid
flowchart LR
    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | c5a1a6b9da15"]
    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | df1b74896efe"]
    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | cb7fd62b858d"]
    artifact3["datastore / cell_selection | filter_pipeline_cells | be30ce93207e"]
    artifact4["RNA / feature_summary | summarize_rna_features | 016854e2c8eb"]
    artifact5["RNA / feature_selection | select_hvgs | 999bfc2e7caf"]
    artifact6["RNA / normalized | run_normalization | dbac6ce55a90"]
    artifact7["RNA / feature_scaling | calculate_feature_scaling | fd9aa6913282"]
    artifact8["RNA / reduction | run_pca | 7af1b0a6d592"]
    artifact9["RNA / ann_index | build_ann_index | b7a5bbdbf02c"]
    artifact10["RNA / neighbors | query_neighbors | 58a19a8550aa"]
    artifact11["RNA / connectivity_map | build_connectivity_map | 269efe4e6156 | outputs: dims20 graph"]
    artifact12["RNA / reduction | run_pca | affdcd24ec99"]
    artifact13["RNA / ann_index | build_ann_index | 87e3df9e75a9"]
    artifact14["RNA / neighbors | query_neighbors | 63cd968f45f8"]
    artifact15["RNA / connectivity_map | build_connectivity_map | 94e23abb785d | outputs: k11 graph"]
    artifact16["RNA / neighbors | query_neighbors | f1cad33ab672"]
    artifact17["RNA / connectivity_map | build_connectivity_map | fcad52813217 | outputs: k15 graph"]
    artifact8 -->|"coordinates"| artifact10
    artifact9 -->|"ann_index"| artifact10
    artifact10 -->|"neighbors"| artifact11
    artifact3 -->|"pca_cell_selection"| artifact12
    artifact6 -->|"normalized"| artifact12
    artifact7 -->|"feature_scaling"| artifact12
    artifact12 -->|"coordinates"| artifact13
    artifact12 -->|"coordinates"| artifact14
    artifact13 -->|"ann_index"| artifact14
    artifact14 -->|"neighbors"| artifact15
    artifact12 -->|"coordinates"| artifact16
    artifact13 -->|"ann_index"| artifact16
    artifact16 -->|"neighbors"| artifact17
    artifact1 -->|"input_cell_selection"| artifact3
    artifact2 -->|"cell_snapshot"| artifact3
    artifact3 -->|"cell_selection"| artifact4
    artifact0 -->|"feature_snapshot"| artifact5
    artifact4 -->|"feature_summary"| artifact5
    artifact3 -->|"cell_selection"| artifact6
    artifact5 -->|"feature_selection"| artifact6
    artifact6 -->|"normalized"| artifact7
    artifact3 -->|"pca_cell_selection"| artifact8
    artifact6 -->|"normalized"| artifact8
    artifact7 -->|"feature_scaling"| artifact8
    artifact8 -->|"coordinates"| artifact9
```

### Artifact details

#### RNA / metadata_snapshot / c5a1a6b9da15
- Status: `complete`
- Path: `RNA/artifacts/metadata_snapshot/c5a1a6b9da152a56b4506813515ef7c4bcd22210ab9ca5e997c5985154402537`
- Operation: `snapshot_run_metadata`
- Parameters: `assay="RNA"; axis="feature"; ordered_columns=["names"]`
- Other inputs: `column_fingerprints={"names":"25b8d03a2f9b2b01183cc9742301f3cf9cd4762c4fe962fe3ad5189013d8c857"}; ordered_row_ids_fingerprint="4548c9820ab9ed0afbc48cbb6503bdcdc725b57c65dbfe3c4da8bb480b7fc4b1"`

#### datastore / cell_selection / df1b74896efe
- Status: `complete`
- Path: `artifacts/cell_selection/df1b74896efe485cbdcbbc889a50a4a7443721c664b1328ae645af657924bf3d`
- Operation: `snapshot_pipeline_input_selection`
- Parameters: `assay="RNA"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="94f668fa448559768ea628c836d7ed1b683636c280cd42e9a6c9b73ba0179a38"`

#### datastore / metadata_snapshot / cb7fd62b858d
- Status: `complete`
- Path: `artifacts/metadata_snapshot/cb7fd62b858d63b8f7f1e9965072ead5c614efa05d07842d9530793ef2de99a3`
- Operation: `snapshot_run_metadata`
- Parameters: `assay=null; axis="cell"; ordered_columns=["names","RNA_nCounts","RNA_nFeatures","RNA_percentMito"]`
- Other inputs: `column_fingerprints={"RNA_nCounts":"a8a7520000e32d28bcf97a8977290bcc7185570098e1fe95739c74687b435843","RNA_nFeatures":"a3df08addee7271194b0ebdfac85ad92ec93f3801031e65316776453eb...; ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"`

#### datastore / cell_selection / be30ce93207e
- Status: `complete`
- Path: `artifacts/cell_selection/be30ce93207e1a708d62a28357f6107c5009649bfdc980aea976219229c8f3f6`
- Operation: `filter_pipeline_cells`
- Parameters: `attrs=["RNA_nCounts","RNA_nFeatures","RNA_percentMito"]; enabled=true; highs=[15000,4000,15]; keepBounds=false; lows=[1000,500,0]; method="manual"`
- Execution options: `source_column="I"`
- Other inputs: `ordered_row_ids_fingerprint="f5f05615ffc8833f75752b75152ccd728702f42950f81e9c28402880a4dd2ae5"; values_fingerprint="6b282098296bdccef544dd1ce41e2f2987a3c75c74240296a5bbbd26e15eb00d"`

#### RNA / feature_summary / 016854e2c8eb
- Status: `complete`
- Path: `RNA/artifacts/feature_summary/016854e2c8eb0bc8b7fe6c223003abd1a3139fb5306bd7c2cf53bad9760b57e2`
- Operation: `summarize_rna_features`
- Parameters: `normalization_method={"module":"scarf.assay","qualname":"norm_lib_size"}; size_factor=1000`
- Execution options: `nthreads=2`

#### RNA / feature_selection / 999bfc2e7caf
- Status: `complete`
- Path: `RNA/artifacts/feature_selection/999bfc2e7caf8a5baf148dcbd6a5a461621b2345f100f1acfe628d69b450628a`
- Operation: `select_hvgs`
- Parameters: `bin_strategy="adaptive"; blacklist="^MT-|^RPS|^RPL|^MRPS|^MRPL|^CCN|^HLA-|^H2-|^HIST|^XIST$|^DDX3Y$|^USP9Y$|^EIF1AY$|^KDM5D$|^SRY$|^ZFY$|^UTY$|^TMSB4Y$|^NLGN4Y$"; keep_bounds=false; lowess_frac=0.1; max_cells=3920; max_mean={"special_float":"inf"}; max_var={"special_float":"inf"}; min_cells=20; min_mean={"special_float":"-inf"}; min_var={"special_float":"-inf"}; n_bins=200; top_n=500; ... 2 more`
- Execution options: `invalidate_cache=false; nthreads=2; plot_kwargs={}; show_plot=false`

#### RNA / normalized / dbac6ce55a90
- Status: `complete`
- Path: `RNA/artifacts/normalized/dbac6ce55a905cd9cf39a8b0f922aea6b19212af357fc5a5c35bd0c431fbfeca`
- Operation: `run_normalization`
- Parameters: `log_transform=true; normalization_method={"external_hook":true,"module":"scarf.assay","qualname":"norm_lib_size"}; renormalize_subset=true; size_factor=1000.0`
- Execution options: `invalidate_cache=false`
- Other inputs: `dataset_fingerprint="ac8346731fc57122b5a0d7b87e7639c5320157483505a3db4e30341b1f902729"`

#### RNA / feature_scaling / fd9aa6913282
- Status: `complete`
- Path: `RNA/artifacts/feature_scaling/fd9aa6913282881a53cf83ccbc91fc1dd9a53f37cfac14699321c9dfdb0773a2`
- Operation: `calculate_feature_scaling`
- Parameters: `enabled=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"`

#### RNA / reduction / 7af1b0a6d592
- Status: `complete`
- Path: `RNA/artifacts/reduction/7af1b0a6d592a9c453e775694eacf744d9a7e2800d7e4db0045444d38ab077b0`
- Operation: `run_pca`
- Parameters: `dims=20; feat_scaling=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / b7a5bbdbf02c
- Status: `complete`
- Path: `RNA/artifacts/ann_index/b7a5bbdbf02cf6a362d8be7f9f4b3a236293b48ea12ece92ecca911fd0f1f55e`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / neighbors / 58a19a8550aa
- Status: `complete`
- Path: `RNA/artifacts/neighbors/58a19a8550aab0daa74351da150ddafdead18a897f8655cf6dda1c018afc66f1`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / 269efe4e6156
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/269efe4e61566bdb32129ec1571f83438686a8af5dbd2aa46b2bdf17ab257db3`
- Operation: `build_connectivity_map`
- Outputs: `dims20 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / reduction / affdcd24ec99
- Status: `complete`
- Path: `RNA/artifacts/reduction/affdcd24ec9900bba36337c6276f847a79f45c2e7fc86de095b6358e39e4d60f`
- Operation: `run_pca`
- Parameters: `dims=15; feat_scaling=true`
- Execution options: `batch_size=3940; invalidate_cache=false; local_cache="auto"; show_elbow_plot=false`

#### RNA / ann_index / 87e3df9e75a9
- Status: `complete`
- Path: `RNA/artifacts/ann_index/87e3df9e75a9b1a79356297ca9ea63eae2e3db546ead572aa935e5d8078d6cef`
- Operation: `build_ann_index`
- Parameters: `ann_ef=50; ann_efc=50; ann_m=48; ann_metric="l2"; ann_parallel=false; parallel_threads=null; rand_state=4466`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / neighbors / 63cd968f45f8
- Status: `complete`
- Path: `RNA/artifacts/neighbors/63cd968f45f8b476dd3dd99b86111bb246ab8f15ca9d8fb16c2d95f518fdafa3`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=11`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / 94e23abb785d
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/94e23abb785d794a6dd3aa79d80f6c20357f5e872a606151e2aff7e8282657d4`
- Operation: `build_connectivity_map`
- Outputs: `k11 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

#### RNA / neighbors / f1cad33ab672
- Status: `complete`
- Path: `RNA/artifacts/neighbors/f1cad33ab672e6af092680eb80e620b4ad3796098c31e3795747a98aff2d9f37`
- Operation: `query_neighbors`
- Parameters: `distance_metric="l2"; k=15`
- Execution options: `batch_size=3940; invalidate_cache=false`

#### RNA / connectivity_map / fcad52813217
- Status: `complete`
- Path: `RNA/artifacts/connectivity_map/fcad52813217774e6c09ecc5829d8d6efd76de44c75d3c19078070f98b6bb6ae`
- Operation: `build_connectivity_map`
- Outputs: `k15 graph`
- Parameters: `bandwidth=1.5; local_connectivity=1.0`
- Execution options: `invalidate_cache=false`

In [7]:
lineage_markdown = lineage.to_markdown()
lineage_markdown.splitlines()[:12]

['```mermaid',
 'flowchart LR',
 '    artifact0["RNA / metadata_snapshot | snapshot_run_metadata | c5a1a6b9da15"]',
 '    artifact1["datastore / cell_selection | snapshot_pipeline_input_selection | df1b74896efe"]',
 '    artifact2["datastore / metadata_snapshot | snapshot_run_metadata | cb7fd62b858d"]',
 '    artifact3["datastore / cell_selection | filter_pipeline_cells | be30ce93207e"]',
 '    artifact4["RNA / feature_summary | summarize_rna_features | 016854e2c8eb"]',
 '    artifact5["RNA / feature_selection | select_hvgs | 999bfc2e7caf"]',
 '    artifact6["RNA / normalized | run_normalization | dbac6ce55a90"]',
 '    artifact7["RNA / feature_scaling | calculate_feature_scaling | fd9aa6913282"]',
 '    artifact8["RNA / reduction | run_pca | 7af1b0a6d592"]',
 '    artifact9["RNA / ann_index | build_ann_index | b7a5bbdbf02c"]']